# Train Joint SID Model for CityLearn

This notebook trains a **joint learned state-space/NARX model** that predicts per-building:

- indoor temperature
- electrical storage SOC
- net electricity consumption
- cooling electricity consumption

It saves artifacts to `saved_files/joint_sid/` for the MPC notebook.

In [ ]:
from pathlib import Path
import sys, os, random, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.INFO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

HERE = Path.cwd()
REPO_DIR = None
for p in [HERE] + list(HERE.parents):
    if (p / "citylearn").exists():
        REPO_DIR = p
        break
if REPO_DIR is None:
    raise FileNotFoundError("Run from inside repo root or set REPO_DIR manually.")

sys.path.insert(0, str(REPO_DIR))
from citylearn.citylearn import CityLearnEnv

# Local helper module from this package
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
from joint_sid_model import normalize_names, obs_to_dicts, safe_get, build_action_vector, build_feature_dict_from_history, STATE_NAMES, TARGET_NAMES, JointResidualMLP

CLIMATE = "TX"
candidates = [
    REPO_DIR / "data" / "datasets" / f"annex96_ce1_{CLIMATE.lower()}_neighborhood",
    REPO_DIR / f"annex96_ce1_{CLIMATE.lower()}_neighborhood",
    REPO_DIR / "annex96_ce1_tx_neighborhood",
]
DATASET_DIR = next((p for p in candidates if (p / "schema.json").exists()), None)
if DATASET_DIR is None:
    raise FileNotFoundError("Could not find TX dataset folder with schema.json")
SCHEMA_PATH = DATASET_DIR / "schema.json"

SID_DIR = REPO_DIR / "saved_files" / "joint_sid"
SID_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_DIR:", REPO_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("SID_DIR:", SID_DIR)

## Environment helpers

In [ ]:
def make_env(central_agent=False):
    return CityLearnEnv(schema=str(SCHEMA_PATH), root_directory=str(DATASET_DIR), central_agent=central_agent)

env = make_env(central_agent=False)
obs, info = env.reset()
OBS_NAMES = normalize_names(env, "observation")
ACTION_NAMES = normalize_names(env, "action")
N_BUILDINGS = len(env.buildings)
print("Buildings:", N_BUILDINGS)
print("Action names b0:", ACTION_NAMES[0])
print("Observation names b0:", OBS_NAMES[0])

## Collect persistent-excitation rollouts

The actions are piecewise random. This is needed because SID needs input excitation. Increase `N_ROLLOUTS` for better models.

In [ ]:
def sample_piecewise_action(env, action_names, rng, previous=None, hold_prob=0.85):
    actions = []
    for b, names in enumerate(action_names):
        low = np.asarray(env.action_space[b].low, dtype=float)
        high = np.asarray(env.action_space[b].high, dtype=float)
        vals = []
        for j, name in enumerate(names):
            if previous is not None and rng.random() < hold_prob:
                vals.append(float(previous[b][j]))
            else:
                if name == "electrical_storage":
                    # avoid constantly extreme battery commands
                    vals.append(float(rng.uniform(low[j], high[j])))
                elif name == "cooling_device":
                    vals.append(float(rng.uniform(low[j], high[j])))
                else:
                    vals.append(0.0)
        actions.append(vals)
    return actions


def collect_rollouts(n_rollouts=8, max_steps=None, seed=42):
    rows = []
    rng = np.random.default_rng(seed)
    for r in range(n_rollouts):
        env = make_env(central_agent=False)
        obs, _ = env.reset()
        obs_names = normalize_names(env, "observation")
        action_names = normalize_names(env, "action")
        n_buildings = len(env.buildings)
        histories = []
        max_lag = max(LAGS)
        obs_ds = obs_to_dicts(obs, obs_names)
        for i, d in enumerate(obs_ds):
            h = {name: [safe_get(d, name, 0.0)] * (max_lag + 1) for name in STATE_NAMES}
            h["action_electrical_storage"] = [0.0] * (max_lag + 1)
            h["action_cooling_device"] = [0.0] * (max_lag + 1)
            histories.append(h)

        prev_actions = None
        t = 0
        while True:
            if max_steps is not None and t >= max_steps:
                break

            current_obs = obs
            current_obs_ds = obs_to_dicts(current_obs, obs_names)
            actions = sample_piecewise_action(env, action_names, rng, previous=prev_actions)
            next_obs, reward, terminated, truncated, info = env.step(actions)
            next_obs_ds = obs_to_dicts(next_obs, obs_names)

            for i in range(n_buildings):
                act_d = {n: float(v) for n, v in zip(action_names[i], actions[i])}
                feat = build_feature_dict_from_history(i, current_obs_ds[i], act_d, histories[i], LAGS)
                y = {
                    "indoor_dry_bulb_temperature_next": safe_get(next_obs_ds[i], "indoor_dry_bulb_temperature", np.nan),
                    "electrical_storage_soc_next": safe_get(next_obs_ds[i], "electrical_storage_soc", np.nan),
                    "net_electricity_consumption_next": safe_get(next_obs_ds[i], "net_electricity_consumption", np.nan),
                    "cooling_electricity_consumption_next": safe_get(next_obs_ds[i], "cooling_electricity_consumption", 0.0),
                }
                row = {**feat, **y, "rollout": r, "t": t, "building": i}
                rows.append(row)

                # update history with next observation and current action
                for name in STATE_NAMES:
                    histories[i][name].append(safe_get(next_obs_ds[i], name, histories[i][name][-1]))
                histories[i]["action_electrical_storage"].append(act_d.get("electrical_storage", 0.0))
                histories[i]["action_cooling_device"].append(act_d.get("cooling_device", 0.0))
                keep = max_lag + 5
                for k in histories[i]:
                    histories[i][k] = histories[i][k][-keep:]

            prev_actions = actions
            obs = next_obs
            t += 1
            if terminated or truncated:
                break
        print(f"rollout {r+1}/{n_rollouts}: {t} steps")
    return pd.DataFrame(rows)

LAGS = (1, 2, 3, 6, 12)
N_ROLLOUTS = 8       # increase to 15-30 for final training
MAX_STEPS = None     # set 200 for quick debug

df = collect_rollouts(n_rollouts=N_ROLLOUTS, max_steps=MAX_STEPS, seed=SEED)
print(df.shape)
df.head()

## Train/validation split

In [ ]:
# Split by rollout, not random rows, to test trajectory generalization.
unique_rollouts = sorted(df["rollout"].unique())
val_rollouts = unique_rollouts[-max(1, len(unique_rollouts)//4):]
train_df = df[~df["rollout"].isin(val_rollouts)].copy()
val_df = df[df["rollout"].isin(val_rollouts)].copy()

ignore = set(TARGET_NAMES + ["rollout", "t", "building"])
feature_names = [c for c in df.columns if c not in ignore]

X_train = train_df[feature_names].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype(np.float32)
Y_train = train_df[TARGET_NAMES].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype(np.float32)
X_val = val_df[feature_names].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype(np.float32)
Y_val = val_df[TARGET_NAMES].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype(np.float32)

x_scaler = StandardScaler()
y_scaler = StandardScaler()
Xs_train = x_scaler.fit_transform(X_train)
Ys_train = y_scaler.fit_transform(Y_train)
Xs_val = x_scaler.transform(X_val)
Ys_val = y_scaler.transform(Y_val)

print("Train:", Xs_train.shape, "Val:", Xs_val.shape)
print("Features:", len(feature_names))

## Linear backbone: Ridge multi-output model

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(Xs_train, Ys_train)

def report_scaled_and_physical(model_pred_scaled, Y_true_phys, label):
    pred_phys = y_scaler.inverse_transform(model_pred_scaled)
    print("\n", label)
    for j, name in enumerate(TARGET_NAMES):
        rmse = mean_squared_error(Y_true_phys[:, j], pred_phys[:, j], squared=False)
        mae = mean_absolute_error(Y_true_phys[:, j], pred_phys[:, j])
        r2 = r2_score(Y_true_phys[:, j], pred_phys[:, j])
        print(f"{name:45s} RMSE={rmse:8.4f} MAE={mae:8.4f} R2={r2:7.4f}")
    return pred_phys

ridge_val_scaled = ridge.predict(Xs_val)
ridge_val_phys = report_scaled_and_physical(ridge_val_scaled, Y_val, "Ridge validation")

## Residual MLP ensemble

In [ ]:
class ArrayDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

BATCH_SIZE = 2048
EPOCHS = 80
PATIENCE = 12
MODEL_CONFIG = {"hidden": 384, "depth": 4, "dropout": 0.05}

train_residual = Ys_train - ridge.predict(Xs_train)
val_residual = Ys_val - ridge.predict(Xs_val)

train_loader = DataLoader(ArrayDataset(Xs_train, train_residual), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ArrayDataset(Xs_val, val_residual), batch_size=BATCH_SIZE, shuffle=False)

def train_one(seed):
    torch.manual_seed(seed)
    model = JointResidualMLP(len(feature_names), len(TARGET_NAMES), **MODEL_CONFIG).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best = 1e18
    best_state = None
    bad = 0
    for epoch in range(EPOCHS):
        model.train()
        tr = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            tr.append(loss.item())
        model.eval()
        va = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE); yb = yb.to(DEVICE)
                va.append(loss_fn(model(xb), yb).item())
        val_loss = float(np.mean(va))
        if val_loss < best:
            best = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
        if epoch % 10 == 0 or epoch == EPOCHS - 1:
            print(f"seed={seed} epoch={epoch:03d} train={np.mean(tr):.5f} val={val_loss:.5f}")
        if bad >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model, best

models = []
for seed in [0, 1, 2]:
    m, best = train_one(SEED + seed)
    models.append(m)
    torch.save(m.state_dict(), SID_DIR / f"joint_residual_mlp_seed_{seed}.pt")
    print("saved", seed, "best", best)

## Evaluate one-step performance

In [ ]:
def ensemble_residual(Xs):
    Xt = torch.tensor(Xs, dtype=torch.float32, device=DEVICE)
    outs = []
    with torch.no_grad():
        for m in models:
            m.eval()
            outs.append(m(Xt).cpu().numpy())
    return np.mean(outs, axis=0)

res_val_scaled = ensemble_residual(Xs_val)
joint_val_scaled = ridge.predict(Xs_val) + res_val_scaled
joint_val_phys = report_scaled_and_physical(joint_val_scaled, Y_val, "Ridge + residual MLP validation")

## Save artifacts

In [ ]:
bundle = {
    "feature_names": feature_names,
    "target_names": TARGET_NAMES,
    "state_names": STATE_NAMES,
    "lags": LAGS,
    "x_scaler": x_scaler,
    "y_scaler": y_scaler,
    "ridge": ridge,
    "model_config": MODEL_CONFIG,
}
joblib.dump(bundle, SID_DIR / "joint_sid_bundle.joblib")
df.to_parquet(SID_DIR / "joint_sid_training_data.parquet", index=False)
print("Saved artifacts to:", SID_DIR)

## Quick open-loop rollout check

This evaluates whether recursive model rollout drifts badly. It uses a validation rollout/building trajectory and recorded actions.

In [ ]:
from joint_sid_model import JointSIDModel

loaded = JointSIDModel(SID_DIR, device=str(DEVICE))

def rollout_check(df, rollout_id=None, building_id=0, max_len=96):
    if rollout_id is None:
        rollout_id = val_rollouts[0]
    part = df[(df.rollout == rollout_id) & (df.building == building_id)].sort_values("t").head(max_len).copy()
    if len(part) < 5:
        print("Not enough rows")
        return
    pred = []
    true = []
    for _, row in part.iterrows():
        feat = {c: float(row[c]) for c in feature_names}
        yhat = loaded.predict_from_feature_dict(feat)
        pred.append([yhat[n] for n in TARGET_NAMES])
        true.append([row[n] for n in TARGET_NAMES])
    pred = np.array(pred); true = np.array(true)
    for j, n in enumerate(TARGET_NAMES):
        print(n, "RMSE", mean_squared_error(true[:,j], pred[:,j], squared=False))
    plt.figure(figsize=(10,4))
    plt.plot(true[:,2], label="true net")
    plt.plot(pred[:,2], label="pred net")
    plt.legend(); plt.title("One-step net electricity check")
    plt.show()

rollout_check(df, building_id=0)